# Agent的基本使用
## 1.Agent中模型的传入方式
### 1.1传入字符串

In [ ]:
from json import tool

from fastmcp.utilities.docstring_parsing import parse_docstring
from langchain.agents import create_agent

from dotenv import load_dotenv
from langchain_tavily import TavilySearch

load_dotenv(override=True)
agent=create_agent(
    model="deepseek:deepseek-v4-flash"
)
print(type(agent))

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

### 1.2 传入模型实例

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

from dotenv import load_dotenv


model=init_chat_model(
    model="deepseek:deepseek-v4-flash"
)
load_dotenv(override=True)
agent=create_agent(
    model=model
)
print(type(agent))

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

## 2.如何调用agent
举例1：

In [ ]:
import os
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # 关键修改：关闭思考模式
    # extra_body={
    #     "thinking": {
    #         "type": "disabled"
    #     }
    # },
)
agent=create_agent(
    model=model
)
response=agent.invoke({
    "messages":[
        {"role":"user","content":"你好"}
    ]
})
rprint(response)

举例2：

In [ ]:
import os
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # 关键修改：关闭思考模式
    # extra_body={
    #     "thinking": {
    #         "type": "disabled"
    #     }
    # },
)
agent=create_agent(
    model=model
)
response=agent.invoke({
    "messages":[
        {"role":"system","content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user","content":"100+20*3=？"}
    ]
})
rprint(response)

## 3.如何绑定工具，并调用工具
### 举例1：绑定一个自定义工具

In [ ]:
import os
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
from langchain_core.tools import  tool
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # 关键修改：关闭思考模式
    # extra_body={
    #     "thinking": {
    #         "type": "disabled"
    #     }
    # },
)
@tool(parse_docstring=True)
def get_weather(city:str):
    """
    查询天气的工具

    Args:
        city:具体的城市
    """
    return f"{city}天气晴朗，温度是15°C"
agent=create_agent(
    model=model,
    tools=[get_weather]
)
response=agent.invoke({
    "messages":[
        {"role":"system","content":"你是一个天气查询助手，请根据用户的提问查询天气，如果问题跟天气无关，你可以说：我不清楚这个问题的答案"},
        {"role":"user","content":"北京的天气如何？"}
    ]
})
rprint(response)

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph().draw_mermaid_png()))

### 举例2：调用langchain内置的工具

In [ ]:
import os
from langchain_tavily import TavilySearch
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
from langchain_core.tools import  tool
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # 关键修改：关闭思考模式
    # extra_body={
    #     "thinking": {
    #         "type": "disabled"
    #     }
    # },
)
#使用内置的工具
web_search=TavilySearch(
    max_results=5,
    tavily_api_key=os.getenv("TAVILY_API_KEY")
)
agent=create_agent(
    model=model,
    tools=[web_search]
)
response=agent.invoke(
    {"messages":
         [
             {"role": "user", "content": "请帮我查询2025年诺贝尔物理学奖得主是谁？"}
        ]
    }
)
rprint(response)

### 举例3：绑定多个工具并调用

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from rich import print as rprint
load_dotenv()
@tool(parse_docstring=True)
def get_weather(city: str):
    """
    天气查询工具

    Args:
    city: 城市名称
    """
    return f"{city}今天天气挺好"
@tool(parse_docstring=True)
def get_news():
    """
    新闻查询工具
    """
    return "近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。"
agent = create_agent(
    model,
    tools=[get_weather, get_news]
)
response = agent.invoke({
    "messages": ["你好，杭州今天的天气如何？今天有哪些新闻？"]
})
rprint(response)